# DUSN-X: fine-tune LLM riêng bằng LoRA
Đây là SFT base Qwen, không phải train từ đầu hoặc train router. Chưa chạy GPU Colab trong repository này.
**Runtime → Change runtime type → T4 GPU**. Chạy lần lượt từng ô. Không đưa token vào ô hoặc output.
Mặc định lưu /content; dữ liệu mất khi ngắt runtime. Sau mỗi chặng phải tải ZIP + SHA xuống máy, Drive chỉ tùy chọn.
Test hiện là bản nháp AI: người thật cần viết/duyệt và khóa trước full train. Không dùng holdout v3 hoặc kết quả của nó.


In [ ]:
import os, sys, subprocess, pathlib, json
REPO_URL = "https://github.com/quocdoanhdang2-sketch/dusnx-platform.git"
REPO_REVISION = "REPLACE_WITH_REVIEWED_DATA_COMMIT"
assert len(REPO_REVISION) == 40 and all(c in "0123456789abcdef" for c in REPO_REVISION.lower()), "Dán SHA commit 40 ký tự chứa dữ liệu đã được người thật duyệt và khóa"
PROJECT = pathlib.Path("/content/dusnx-platform")
RUN = pathlib.Path("/content/dusnx-llm-run")
if not PROJECT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT)], check=True)
subprocess.run(["git", "-C", str(PROJECT), "checkout", "--detach", REPO_REVISION], check=True)
os.chdir(PROJECT)
CODE_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Repository commit:", CODE_COMMIT, "Python:", sys.version)
assert (3, 11) <= sys.version_info[:2] <= (3, 13), "Use Python 3.11–3.13 for the pinned stack"
os.environ["PYTHONPATH"] = str(PROJECT / "python/src")
LLM_ENV = pathlib.Path("/content/dusnx-llm-env")
if not LLM_ENV.exists():
    subprocess.run([sys.executable, "-m", "venv", str(LLM_ENV)], check=True)
LLM_PY = LLM_ENV / "bin/python"
sys.path.insert(0, str(PROJECT / "python/src"))
def call(*args):
    subprocess.run([str(LLM_PY), *map(str, args)], check=True)


Package được cài trong venv /content/dusnx-llm-env riêng để không xung đột torchvision/torchaudio có sẵn của Colab. Kernel vẫn dùng google.colab để download. Không dùng Run all khi chưa duyệt dữ liệu.

In [ ]:
call("-m", "pip", "install", "-r", "requirements/llm-sft.txt")
call("-c", "import torch; print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available()); print(torch.cuda.get_device_properties(0) if torch.cuda.is_available() else 'CPU smoke only; full SFT requires GPU')")
HAS_GPU = subprocess.check_output([str(LLM_PY), "-c", "import torch; print(int(torch.cuda.is_available()))"], text=True).strip() == "1"


In [ ]:
call("python/scripts/prepare_llm_data.py", "--output", "/content/llm-audit", "--tokenizer-model", "Qwen/Qwen2.5-0.5B-Instruct", "--tokenizer-revision", "7ae557604adf67be50417f59c2c2f167def9a775", "--max-length", "512")
print(pathlib.Path("/content/llm-audit/audit.json").read_text())
print("Đọc human_review_sample.json và datasets/llm_sft trước full train.")


Smoke CPU dùng Qwen rất nhỏ khởi tạo ngẫu nhiên, không tải pretrained model; kiểm tra loss mask, LoRA, save và resume. Kết quả này không chứng minh chất lượng Qwen fine-tuned.

In [ ]:
SMOKE = pathlib.Path("/content/llm-smoke")
call("python/scripts/finetune_llm.py", "--smoke", "--output", SMOKE, "--stop-after-epoch", "1")
last = sorted(SMOKE.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))[-1]
call("python/scripts/finetune_llm.py", "--smoke", "--output", SMOKE, "--resume", last)


### Chốt dữ liệu rồi train theo chặng
Không chọn cấu hình theo test. Hãy đọc tài liệu WEEK3_LLM_TRAINING.md, duyệt train/validation và nhờ người viết/duyệt test riêng, khóa SHA trước khi bật RUN_FULL. Không sửa test sau khi xem prediction.
Config mặc định batch=1, accumulation=8, length=512, LoRA r=8; validation chọn checkpoint tốt nhất.
Nếu OOM: kiểm tra token lengths, dùng config mới với length phù hợp hoặc qlora=true + requirements/llm-qlora.txt. Run mới khi đổi config; không resume khác cấu hình.


In [ ]:
RUN_FULL = False  # Chỉ bật sau khi kiểm tra dữ liệu và có GPU.
CONFIG = "configs/llm_sft.yaml"
if RUN_FULL:
    assert HAS_GPU, "Full train cần GPU"
    lock = json.loads(pathlib.Path("datasets/llm_sft/manifest.json").read_text())
    assert lock.get("status") == "human_attested_locked_before_training", "Dữ liệu chưa được con người duyệt/khóa"
    assert pathlib.Path("datasets/llm_sft/human_test_attestation.json").is_file(), "Thiếu attestation test độc lập do người viết"
    call("python/scripts/finetune_llm.py", "--config", CONFIG, "--output", RUN, "--stop-after-epoch", "1")
else:
    print("Full pretrained SFT chưa được chạy.")


Tải chặng 1 ngay. Trình duyệt có thể hỏi cho phép nhiều lượt tải. Kiểm tra cả ZIP và SHA đã có trong Downloads. Tiếp tục chặng 2/3 chỉ khi đã giữ bản backup ngoài /content.

In [ ]:
from google.colab import files
if RUN_FULL:
    files.download(str(RUN.with_suffix(".zip")))
    files.download(str(RUN.with_suffix(".zip.sha256")))


In [ ]:
NEXT_EPOCH = 2  # Chạy ô này và ô download sau đó; đổi thành 3 để chạy chặng cuối.
if RUN_FULL:
    last = sorted(RUN.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))[-1]
    call("python/scripts/finetune_llm.py", "--config", CONFIG, "--output", RUN, "--resume", last, "--stop-after-epoch", str(NEXT_EPOCH))
    files.download(str(RUN.with_suffix(".zip")))
    files.download(str(RUN.with_suffix(".zip.sha256")))


### Khôi phục sau runtime mất (bỏ qua nếu vẫn còn RUN)
Trong phiên Colab mới, chạy ô clone/cài package; tải lên ZIP cùng SHA của chặng gần nhất. Không nhập checksum bịa. Chỉ giải nén vào thư mục trống; helper chặn path traversal. Sau đó bật RUN_FULL và chạy ô NEXT_EPOCH kế tiếp, **không chạy ô train chặng 1**.


In [ ]:
RESTORE = False
if RESTORE:
    uploaded = files.upload()
    zips = [pathlib.Path(n) for n in uploaded if n.endswith(".zip")]
    assert len(zips) == 1, "Upload đúng một ZIP và file .zip.sha256 của nó"
    archive = zips[0]
    sha = archive.with_suffix(".zip.sha256").read_text().strip()
    call("python/scripts/export_llm.py", "unpack", "--zip", archive, "--sha256", sha, "--output", RUN)


Drive tùy chọn: nếu mount lỗi, tiếp tục dùng ZIP. Không phụ thuộc Drive để train.

In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount("/content/drive")
    backup = pathlib.Path("/content/drive/MyDrive/dusnx-llm-backup")
    backup.mkdir(parents=True, exist_ok=True)
    for p in RUN.parent.glob(RUN.name + "*.zip*"):
        shutil.copy2(p, backup / p.name)


### Merge và GGUF
Chỉ sau full train hoàn tất. Merge CPU float32 cần RAM để chứa base + adapter; F16 GGUF khoảng 1 GB, merged/ZIP lớn hơn. Converter pin llama.cpp b6500 bằng SHA. Lỗi converter hoặc thiếu RAM là export chưa đạt, không đổi định dạng giả.


In [ ]:
EXPORT = pathlib.Path("/content/dusnx-llm-export")
RUN_EXPORT = False
if RUN_EXPORT:
    call("python/scripts/export_llm.py", "merge-gguf", "--run", RUN, "--output", EXPORT, "--converter", "/content/llama.cpp")
    files.download(str(EXPORT.with_suffix(".zip")))
    files.download(str(EXPORT.with_suffix(".zip.sha256")))


### Đánh giá cùng prompt
Tải run ZIP và export ZIP về Windows. Làm các lệnh import + paired Gateway evaluation trong docs/WEEK3_LLM_TRAINING.md. Model candidate chưa tồn tại trước full train; notebook không tạo điểm so sánh giả. Bảng blind_ratings.csv cần người thật chấm cả hai model; raw_outputs.json có provenance LLM và độ trễ.
So sánh HF trước export (ô tùy chọn dưới) chỉ chạy sau test được người viết/duyệt và khóa; không dùng điểm test để chọn lại cấu hình. So sánh triển khai chính thức vẫn phải đi qua Gateway/Ollama vì lượng tử hóa/template có thể làm đổi đáp án.


In [ ]:
RUN_HF_EVAL = False
if RUN_HF_EVAL:
    call("python/scripts/evaluate_llm_hf.py", "--run", RUN, "--output", "/content/llm-hf-comparison")
    call("python/scripts/export_llm.py", "pack", "--root", "/content/llm-hf-comparison", "--zip", "/content/llm-hf-comparison.zip")
    files.download("/content/llm-hf-comparison.zip")
    files.download("/content/llm-hf-comparison.zip.sha256")
